# CUTEst

In [1]:
import multiprocessing as mp
import os
import subprocess
import sys
from concurrent.futures import ProcessPoolExecutor, as_completed
from pathlib import Path

# Each CUTEst task runs in a separate process. Keep BLAS single-threaded to avoid
# oversubscribing the machine when several worker processes run simultaneously.
for variable in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[variable] = "1"

working_directory = Path.cwd().resolve()
if (working_directory / "pyproject.toml").exists():
    repository_root = working_directory
elif (working_directory.parent / "pyproject.toml").exists():
    repository_root = working_directory.parent
else:
    raise RuntimeError("Open this notebook from the repository root or notebooks/.")
if not os.environ.get("PYCUTEST_CACHE"):
    os.environ["PYCUTEST_CACHE"] = str(
        repository_root
        / "submodules"
        / "paper-regularized-qn-benchmark"
        / "pycutest_cache_holder"
    )
pycutest_cache_root = Path(os.environ["PYCUTEST_CACHE"])
pycutest_cache_root.mkdir(parents=True, exist_ok=True)
(pycutest_cache_root / "pycutest_cache_holder").mkdir(
    parents=True, exist_ok=True
)

import numpy as np
import pandas as pd

from data.CUTEst.check_CUTEst_problems import problemsToRun
from qnlab.experiment.for_cutest_run import (
    CUTEstTask,
    get_file_path,
    load_npz_with_metadata,
    result_matches_task,
    run_tasks,
    scaled_iteration_limit,
    solve_problem_with_timeout,
)
from qnlab.experiment.for_cutest_vis import draw_data_profile, draw_pp
from qnlab.util.method import COLORS, LINE_STYLES, Method, get_methods

## Experiment workflow

This single workflow covers the floating-point experiments and all explicit-noise experiments reported in the paper. Results are stored under `data/temp/<scenario>/seed_<seed>/`. SciPy runs first with at most 15,000 iterations. If SciPy reaches the scenario tolerance after `k` iterations, each other solver receives at most `min(15,000, 15k)` iterations; otherwise it receives 15,000. Independent tasks run in separate worker processes because CUTEst problems should not share mutable Fortran state within one process. Before parallel execution, any uncached CUTEst interfaces are built sequentially in the repository cache to avoid concurrent PyCUTEst cache creation. `MAX_WORKERS` controls the parallelism and defaults to at most 8; set it to 1 for sequential execution. Start from a fresh kernel so that the cache and single-threaded BLAS settings in the import cell propagate to every worker. With `OVERWRITE_EXISTING = False`, a stored result is reused only when its task-defining metadata exactly matches the current task, so an interrupted experiment can be continued without mixing incompatible settings. If any worker fails, the cell stops after the batch and reports the failure; rerunning the cell reuses all matching completed results. With `OVERWRITE_EXISTING = True`, every selected task is rerun and its existing result is replaced. A full run is long-running.

In [2]:
os.chdir(repository_root)
print(repository_root)
print(f"PYCUTEST_CACHE={os.environ['PYCUTEST_CACHE']}")

/home/hirok/University/qnlab


In [3]:
PRECISION_FUNCTION_ERRORS = {
    64: np.finfo(np.float64).eps * 1e7,
    32: np.finfo(np.float32).eps * 1e4,
    16: np.finfo(np.float16).eps * 1e2,
}

SCENARIOS = {
    "float64": {
        "precision": 64,
        "function_noise": 0.0,
        "gradient_noise": 0.0,
        "assumed_function_error": PRECISION_FUNCTION_ERRORS[64],
        "solver_gtol": 1e-5,
        "score_gtols": [1e-1, 1e-3, 1e-5],
    },
    "float32": {
        "precision": 32,
        "function_noise": 0.0,
        "gradient_noise": 0.0,
        "assumed_function_error": PRECISION_FUNCTION_ERRORS[32],
        "solver_gtol": 1e-5,
        "score_gtols": [1e-1, 1e-3, 1e-5],
    },
    "float16": {
        "precision": 16,
        "function_noise": 0.0,
        "gradient_noise": 0.0,
        "assumed_function_error": PRECISION_FUNCTION_ERRORS[16],
        "solver_gtol": 1e-5,
        "score_gtols": [1e-1, 1e-3, 1e-5],
    },
    # Theory-aligned experiment: the gradient oracle is exact.
    "function_only": {
        "precision": 64,
        "function_noise": 1e-3,
        "gradient_noise": 0.0,
        "assumed_function_error": 1e-2,
        "solver_gtol": 1e-2,
        "score_gtols": [1e-2],
    },
    # Deliberately adverse stress test outside the exact-gradient theorem.
    "joint_noise": {
        "precision": 64,
        "function_noise": 1e-3,
        "gradient_noise": 1e-3,
        "assumed_function_error": 1e-2,
        "solver_gtol": 1e-2,
        "score_gtols": [1e-2],
    },
    "eps_under": {
        "precision": 64,
        "function_noise": 1e-3,
        "gradient_noise": 0.0,
        "assumed_function_error": 1e-4,
        "solver_gtol": 1e-2,
        "score_gtols": [1e-2],
    },
    "eps_nominal": {
        "precision": 64,
        "function_noise": 1e-3,
        "gradient_noise": 0.0,
        "assumed_function_error": 1e-2,
        "solver_gtol": 1e-2,
        "score_gtols": [1e-2],
    },
    "eps_over": {
        "precision": 64,
        "function_noise": 1e-3,
        "gradient_noise": 0.0,
        "assumed_function_error": 1e-1,
        "solver_gtol": 1e-2,
        "score_gtols": [1e-2],
    },
}

# SAFETY: the checked-in default is an intentionally tiny pilot, never the full suite.
MEMORY_SIZE = 10
OFFO_SQUARED_OFFSET = 1e-20
ASTR1_ADAGRAD_VARSIGMA = 1e-2
ASTR1_ADAGRAD_THETA = 1.0
RESTART_THRESHOLD = 1.0
MAX_RESTARTS = 10
SCENARIOS_TO_RUN = [
    "float64",
    "float32",
    "float16",
    "function_only",
    "joint_noise",
    "eps_under",
    "eps_over",
    # "eps_nominal" is unnecessary
]
RESULT_SCENARIO_ALIASES = {"eps_nominal": "function_only"}
NOISY_SEEDS = list(range(5))
NOISY_SEEDS = NOISY_SEEDS[:1]  # Pilot: comment out this line for all five seeds.
PROBLEMS_TO_RUN = 50
METHODS_TO_RUN = None
TIME_LIMIT = 600.0
MAX_ITERATIONS = 15000
MAX_WORKERS = min(8, os.cpu_count() or 1)  # Set to 1 for sequential execution.
OVERWRITE_EXISTING = False
ERROR_CAUSING_TASKS = [
    (16, "INDEFM", "SciPy"),
    (16, "INDEFM", "NTRQN"),
    (16, "INDEFM", "NTRQN-MS"),
    (16, "INDEFM", "Reg-Sec"),
    (16, "OSCIGRAD", "NTRQN-MS"),
    (32, "INDEFM", "NTRQN"),
    (32, "INDEFM", "NTRQN-MS"),
    (32, "OSCIGRAD", "NTRQN-MS"),
]

## Methods and execution


In [4]:
STANDARD_LABELS = {
    "NTRQN",
    "NTRQN-MS",
    "Line",
    "Line-MS",
    "ASTR1-Adagrad",
    "Reg",
    "Reg-Sec",
    "SciPy",
    "NTQN",
}
SCENARIO_DEFAULT_LABELS = {
    "function_only": STANDARD_LABELS
    | {"NTRQN-Restart", "NTQN-Default-Termination"},
    "joint_noise": STANDARD_LABELS | {"NTQN-Default-Termination"},
    "eps_under": {"NTRQN", "NTRQN-MS"},
    "eps_nominal": {"NTRQN", "NTRQN-MS"},
    "eps_over": {"NTRQN", "NTRQN-MS"},
}
for scenario in ("float64", "float32", "float16"):
    SCENARIO_DEFAULT_LABELS[scenario] = STANDARD_LABELS

In [5]:
def warm_cutest_cache(tasks):
    cache_entries = pycutest_cache_root / "pycutest_cache_holder"
    problem_names = sorted({task.problem_name for task in tasks})
    missing = [name for name in problem_names if not (cache_entries / name).is_dir()]
    if not missing:
        return
    print(f"Preparing {len(missing)} uncached CUTEst problem(s) sequentially.")
    command = (
        "import pycutest, sys; "
        "pycutest.import_problem(sys.argv[1])"
    )
    for index, problem_name in enumerate(missing, start=1):
        subprocess.run(
            [sys.executable, "-c", command, problem_name],
            check=True,
            env=os.environ.copy(),
        )
        print(f"[{index}/{len(missing)}] Cached: {problem_name}")


def run_tasks_with_workers(
    tasks, error_causing_tasks, time_limit, result_subdir=None, overwrite=False
):
    if MAX_WORKERS <= 1:
        return run_tasks(
            tasks, error_causing_tasks, time_limit,
            result_subdir=result_subdir, overwrite=overwrite,
        )
    pending = [
        task for task in tasks
        if overwrite or not result_matches_task(task, result_subdir)
    ]
    print(f"Total tasks to run: {len(pending)} with up to {MAX_WORKERS} workers")
    if not pending:
        return
    warm_cutest_cache(pending)
    known_errors = set(error_causing_tasks)
    errors = []
    context = mp.get_context("fork")
    with ProcessPoolExecutor(
        max_workers=min(MAX_WORKERS, len(pending)), mp_context=context
    ) as executor:
        future_to_task = {}
        for task in pending:
            known_error = (
                task.precision, task.problem_name, task.method.label
            ) in known_errors
            task_time_limit = 60 if known_error else time_limit
            if known_error:
                print(
                    f"⚠ Reducing time limit for known error-causing task: "
                    f"{task.problem_name} with {task.method.label}"
                )
            future = executor.submit(
                solve_problem_with_timeout,
                task,
                task_time_limit,
                known_error or time_limit >= 600,
                result_subdir,
            )
            future_to_task[future] = task
        for completed, future in enumerate(as_completed(future_to_task), start=1):
            task = future_to_task[future]
            try:
                future.result()
                print(
                    f"[{completed}/{len(pending)}] ✓ Finished: "
                    f"{task.problem_name} with {task.method.label}"
                )
            except Exception as error:
                file_path = get_file_path(task, result_subdir)
                print(f"[{completed}/{len(pending)}] ⚠ Error: {file_path}: {error!r}")
                errors.append(file_path)
    if errors:
        raise RuntimeError(
            f"{len(errors)} CUTEst task(s) failed. Fix the reported error and "
            "rerun this cell; matching completed results will be reused."
        )


def get_experiment_methods(max_iterations, solver_gtol=None):
    common = {"m": MEMORY_SIZE, "max_iterations": max_iterations}
    methods = [
        (
            Method("NTRQN", "cautious", "damped", "bfgs", label="NTRQN"),
            common
            | {
                "offo_squared_offset": OFFO_SQUARED_OFFSET,
                "max_restarts": 0,
            },
        ),
        (
            Method("NTRQN", "cautious", "damped_modified", "bfgs", label="NTRQN-MS"),
            common
            | {
                "offo_squared_offset": OFFO_SQUARED_OFFSET,
                "max_restarts": 0,
            },
        ),
        (Method("Line", "raw", "raw", "bfgs", label="Line"), common),
        (
            Method("Line", "raw", "modified", "bfgs", label="Line-MS"),
            common,
        ),
        (Method("Kanzow", "raw", "raw", "bfgs", label="Reg"), common),
        (
            Method("KanzowSec", "raw", "raw", "bfgs", label="Reg-Sec"),
            common,
        ),
        (
            Method("SciPy", scipy_method="L-BFGS-B", label="SciPy"),
            {"maxcor": MEMORY_SIZE, "maxiter": max_iterations, "ftol": 0},
        ),
        (
            Method("NTQN", "raw", "raw", "bfgs", label="NTQN"),
            common | {"terminate": 1, "stop_at_gtol": 1},
        ),
        (
            Method("OFFO", label="ASTR1-Adagrad"),
            {
                "max_iterations": max_iterations,
                "varsigma": ASTR1_ADAGRAD_VARSIGMA,
                "theta": ASTR1_ADAGRAD_THETA,
            },
        ),
        (
            Method("NTRQN", "cautious", "damped", "bfgs", label="NTRQN-Restart"),
            common
            | {
                "offo_squared_offset": OFFO_SQUARED_OFFSET,
                "restart_threshold": RESTART_THRESHOLD,
                "max_restarts": MAX_RESTARTS,
            },
        ),
        (
            Method("NTQN", "raw", "raw", "bfgs", label="NTQN-Default-Termination"),
            common | {"terminate": 3, "stop_at_gtol": 0},
        ),
    ]
    if solver_gtol is not None:
        methods = [
            (method, option | {"gtol": solver_gtol}) for method, option in methods
        ]
    return methods


def get_scipy_method(solver_gtol):
    methods = get_experiment_methods(MAX_ITERATIONS, solver_gtol)
    return next(entry for entry in methods if entry[0].label == "SciPy")


def get_instance_iteration_limit(scenario, seed, problem_name):
    source_scenario = RESULT_SCENARIO_ALIASES.get(scenario, scenario)
    config = SCENARIOS[source_scenario]
    scipy_method, scipy_options = get_scipy_method(config["solver_gtol"])
    scipy_task = make_task(
        source_scenario, seed, problem_name, scipy_method, scipy_options
    )
    callback, _ = load_npz_with_metadata(scipy_task, verbose=False)
    return scaled_iteration_limit(
        callback, config["solver_gtol"], maximum=MAX_ITERATIONS
    )


def get_instance_methods(scenario, seed, problem_name):
    config = SCENARIOS[scenario]
    iteration_limit = get_instance_iteration_limit(scenario, seed, problem_name)
    methods = get_experiment_methods(iteration_limit, config["solver_gtol"])
    scipy_method = get_scipy_method(config["solver_gtol"])
    return [scipy_method if method.label == "SciPy" else (method, options)
            for method, options in methods]


def get_scenario_seeds(scenario):
    config = SCENARIOS[scenario]
    has_noise = config["function_noise"] > 0.0 or config["gradient_noise"] > 0.0
    return NOISY_SEEDS if has_noise else [0]


def make_task(scenario, seed, problem_name, method, options):
    config = SCENARIOS[scenario]
    assumed_error = config["assumed_function_error"]
    return CUTEstTask(
        problem_name=problem_name,
        method=method,
        options=options,
        precision=config["precision"],
        function_noise=np.float64(config["function_noise"]),
        gradient_noise=np.float64(config["gradient_noise"]),
        assumed_function_error=(
            None if assumed_error is None else np.float64(assumed_error)
        ),
        seed=seed,
        scenario=scenario,
    )


def result_source(scenario, seed, method):
    source_scenario = RESULT_SCENARIO_ALIASES.get(scenario, scenario)
    assert source_scenario in SCENARIOS, f"Unknown scenario: {source_scenario}"
    return source_scenario, seed


def load_scenario_results(scenario, seeds, gtol, labels=None):
    config = SCENARIOS[scenario]
    problems = problemsToRun(config["precision"])
    if type(PROBLEMS_TO_RUN) is int:
        problems = problems[:PROBLEMS_TO_RUN]
    labels = labels or METHODS_TO_RUN or SCENARIO_DEFAULT_LABELS[scenario]
    all_methods = get_experiment_methods(MAX_ITERATIONS, config["solver_gtol"])
    alg_names = [method.label for method, _ in all_methods if method.label in labels]
    instances = [(seed, problem) for seed in seeds for problem in problems]
    calls = np.full((len(alg_names), len(instances)), np.inf)
    dimensions = np.full(len(instances), np.nan)
    metadata = np.empty((len(alg_names), len(instances)), dtype=object)
    metadata.fill(None)
    for instance_index, (seed, problem) in enumerate(instances):
        method_options = get_instance_methods(scenario, seed, problem)
        method_options = [
            entry for entry in method_options if entry[0].label in labels
        ]
        for method_index, (method, options) in enumerate(method_options):
            source_scenario, source_seed = result_source(scenario, seed, method)
            task = make_task(source_scenario, source_seed, problem, method, options)
            callback, run_metadata = load_npz_with_metadata(task, verbose=False)
            metadata[method_index, instance_index] = run_metadata
            if "dimension" in run_metadata:
                dimensions[instance_index] = run_metadata["dimension"]
            reached = np.flatnonzero(np.asarray(callback.gnorms) <= gtol)
            if reached.size > 0:
                calls[method_index, instance_index] = max(1, callback.calls[reached[0]])
    instance_names = [f"{problem} (seed={seed})" for seed, problem in instances]
    return alg_names, calls, instance_names, dimensions, metadata


def summarize_run_metadata(alg_names, metadata):
    rows = []
    for method_index, method in enumerate(alg_names):
        entries = [entry for entry in metadata[method_index] if entry]
        statuses = [entry["status"] for entry in entries]
        return_values = [entry.get("return_code_value") for entry in entries]
        rows.append(
            {
                "method": method,
                "result files": len(entries),
                "missing files": metadata.shape[1] - len(entries),
                "completed": statuses.count("completed"),
                "timeouts": statuses.count("timeout"),
                "exceptions": statuses.count("error"),
                "error return codes": sum(
                    value is not None and value < 0 for value in return_values
                ),
            }
        )
    return pd.DataFrame(rows).set_index("method")


def summarize_diagnostics(alg_names, metadata):
    restart_rows = []
    ntqn_rows = []
    for method_index, method in enumerate(alg_names):
        diagnostics = [
            entry.get("diagnostics", {}) for entry in metadata[method_index] if entry
        ]
        if method == "NTRQN-Restart" and diagnostics:
            counts = np.asarray(
                [entry.get("OFFO accumulator restart", 0) for entry in diagnostics]
            )
            restart_rows.append(
                {
                    "method": method,
                    "mean": counts.mean(),
                    "median": np.median(counts),
                    "maximum": counts.max(),
                    "cap reached": np.count_nonzero(counts >= MAX_RESTARTS),
                }
            )
        for entry in diagnostics:
            if "NTQN termination flag" in entry:
                ntqn_rows.append(
                    {"method": method, "flag": entry["NTQN termination flag"]}
                )
    restart_summary = pd.DataFrame(restart_rows)
    ntqn_summary = (
        pd.crosstab(
            pd.Series([row["method"] for row in ntqn_rows], name="method"),
            pd.Series([row["flag"] for row in ntqn_rows], name="flag"),
        )
        if ntqn_rows
        else pd.DataFrame()
    )
    return restart_summary, ntqn_summary

In [6]:
unknown_scenarios = sorted(set(SCENARIOS_TO_RUN) - set(SCENARIOS))
if unknown_scenarios:
    raise ValueError(f"Unknown scenarios: {unknown_scenarios}")

scipy_tasks = []
instances_by_scenario = {}
for scenario in SCENARIOS_TO_RUN:
    config = SCENARIOS[scenario]
    selected_problems = problemsToRun(config["precision"])
    if type(PROBLEMS_TO_RUN) is int:
        selected_problems = selected_problems[:PROBLEMS_TO_RUN]
    instances = [
        (seed, problem)
        for seed in get_scenario_seeds(scenario)
        for problem in selected_problems
    ]
    instances_by_scenario[scenario] = instances
    scipy_method, scipy_options = get_scipy_method(config["solver_gtol"])
    scipy_tasks.extend(
        make_task(scenario, seed, problem, scipy_method, scipy_options)
        for seed, problem in instances
    )

print(f"Scenarios: {SCENARIOS_TO_RUN}")
print(f"Prepared {len(scipy_tasks)} SciPy reference tasks.")
run_tasks_with_workers(
    scipy_tasks,
    ERROR_CAUSING_TASKS,
    int(TIME_LIMIT),
    overwrite=OVERWRITE_EXISTING,
)

tasks = []
reused_task_count = 0
for scenario in SCENARIOS_TO_RUN:
    labels = METHODS_TO_RUN or SCENARIO_DEFAULT_LABELS[scenario]
    known_labels = {
        method.label
        for method, _ in get_experiment_methods(MAX_ITERATIONS)
    }
    missing = sorted(set(labels) - known_labels)
    if missing:
        raise ValueError(f"Unknown method labels: {missing}")
    for seed, problem in instances_by_scenario[scenario]:
        selected_method_options = get_instance_methods(scenario, seed, problem)
        selected_method_options = [
            entry for entry in selected_method_options
            if entry[0].label in labels and entry[0].label != "SciPy"
        ]
        for method, options in selected_method_options:
            source_scenario, source_seed = result_source(scenario, seed, method)
            if (source_scenario, source_seed) != (scenario, seed):
                reused_task_count += 1
                continue
            tasks.append(make_task(scenario, seed, problem, method, options))

print(f"Prepared {len(tasks)} non-SciPy tasks.")
if reused_task_count:
    print(f"Reusing {reused_task_count} equivalent stored results.")

run_tasks_with_workers(
    tasks, ERROR_CAUSING_TASKS, int(TIME_LIMIT), overwrite=OVERWRITE_EXISTING
)

Scenarios: ['float64', 'float32', 'float16', 'function_only', 'joint_noise', 'eps_under', 'eps_over']
Prepared 2400 tasks.
Total tasks to run: 0


## Visualize saved results

Select scenarios after their runs have completed. The default empty list avoids overwriting existing figures accidentally.

In [ ]:
SCENARIOS_TO_PLOT = [
    "float64",
    "float32",
    "float16",
    "function_only",
    "joint_noise",
    "eps_under",
    "eps_nominal",
    "eps_over",
]


def display_result_summaries(alg_names, calls, metadata):
    best_calls = calls.min(axis=0)
    fastest = (calls == best_calls) & np.isfinite(best_calls)[None, :]
    display(
        pd.DataFrame(
            {
                "solved (percent)": 100.0 * np.isfinite(calls).mean(axis=1),
                "fastest, ties included (percent)": 100.0 * fastest.mean(axis=1),
            },
            index=alg_names,
        )
    )
    display(summarize_run_metadata(alg_names, metadata))
    restart_summary, ntqn_summary = summarize_diagnostics(alg_names, metadata)
    if not restart_summary.empty:
        display(restart_summary.set_index("method"))
    if not ntqn_summary.empty:
        display(ntqn_summary)


_, ALGORITHM_COLORS, ALGORITHM_LINE_STYLES = get_methods()
ALGORITHM_COLORS["ASTR1-Adagrad"] = COLORS["ASTR1-Adagrad"]
ALGORITHM_LINE_STYLES["ASTR1-Adagrad"] = LINE_STYLES["ASTR1-Adagrad"]
for scenario in SCENARIOS_TO_PLOT:
    config = SCENARIOS[scenario]
    for gtol in config["score_gtols"]:
        seeds = get_scenario_seeds(scenario)
        primary_labels = METHODS_TO_RUN or (
            SCENARIO_DEFAULT_LABELS[scenario] - {"NTRQN-Restart"}
        )
        alg_names, calls, instances, dimensions, metadata = load_scenario_results(
            scenario, seeds, gtol, labels=primary_labels
        )
        output_path = None
        if scenario not in {"float64", "float32", "float16", "joint_noise"}:
            gtol_name = f"{gtol:.0e}".replace("+", "")
            output_path = (
                Path("doc/imgs/compare") / f"_pp_{scenario}_gtol{gtol_name}.pdf"
            )
        draw_pp(
            alg_names,
            calls,
            ALGORITHM_COLORS,
            ALGORITHM_LINE_STYLES,
            config["precision"],
            np.float64(max(config["function_noise"], config["gradient_noise"])),
            np.float64(gtol),
            output_path=output_path,
        )
        gtol_name = f"{gtol:.0e}".replace("+", "")
        data_profile_path = (
            Path("doc/imgs/compare") / f"_dp_{scenario}_gtol{gtol_name}.pdf"
        )
        if np.all(np.isfinite(dimensions)):
            draw_data_profile(
                alg_names,
                calls,
                dimensions,
                ALGORITHM_COLORS,
                ALGORITHM_LINE_STYLES,
                data_profile_path,
            )
        else:
            print("Skipping data profile because some result metadata lack dimensions.")

        table = pd.DataFrame(calls.T, index=instances, columns=alg_names)
        display(table)

        display_result_summaries(alg_names, calls, metadata)

        if len(seeds) > 1:
            solved_by_seed = []
            for seed in seeds:
                _, seed_calls, _, _, _ = load_scenario_results(
                    scenario, [seed], gtol, labels=primary_labels
                )
                solved_by_seed.append(np.isfinite(seed_calls).mean(axis=1))
            solved_by_seed = np.asarray(solved_by_seed)
            seed_summary = pd.DataFrame(
                {
                    "mean solved fraction": solved_by_seed.mean(axis=0),
                    "std. dev. across seeds": solved_by_seed.std(axis=0, ddof=1),
                },
                index=alg_names,
            )
            display(seed_summary)

        if scenario == "function_only" and METHODS_TO_RUN is None:
            restart_labels = {"NTRQN", "NTRQN-Restart"}
            restart_names, restart_calls, _, restart_dimensions, restart_metadata = (
                load_scenario_results(scenario, seeds, gtol, labels=restart_labels)
            )
            restart_stem = f"function_only_restart_gtol{gtol_name}"
            draw_pp(
                restart_names,
                restart_calls,
                ALGORITHM_COLORS,
                ALGORITHM_LINE_STYLES,
                config["precision"],
                np.float64(config["function_noise"]),
                np.float64(gtol),
                output_path=Path("doc/imgs/compare") / f"_pp_{restart_stem}.pdf",
            )
            if np.all(np.isfinite(restart_dimensions)):
                draw_data_profile(
                    restart_names,
                    restart_calls,
                    restart_dimensions,
                    ALGORITHM_COLORS,
                    ALGORITHM_LINE_STYLES,
                    Path("doc/imgs/compare") / f"_dp_{restart_stem}.pdf",
                )
            display_result_summaries(restart_names, restart_calls, restart_metadata)